# Preprocessing: Feature Contract Alignment


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

TARGET_COLS = [
    'Total Alkalinity',
    'Electrical Conductance',
    'Dissolved Reactive Phosphorus',
]

GEO_TIME_COLS = ['Latitude', 'Longitude', 'Sample Date']
SANLC_PREFIXES = ('sanlc2020_pct_', 'sanlc2022_pct_')

TRAIN_PATH = Path('../data/interim/master_train_iteration.parquet')
TEST_PATH = Path('../data/interim/master_test_iteration.parquet')

CONTRACT_TXT_PATH = Path('../data/interim/feature_contract_master_iteration.txt')
CONTRACT_META_PATH = Path('../data/interim/feature_contract_master_iteration_meta.json')

OUT_TRAIN_PATH = Path('../data/interim/master_train_iteration_aligned.parquet')
OUT_TEST_PATH = Path('../data/interim/master_test_iteration_aligned.parquet')
OUT_REPORT_PATH = Path('../data/interim/feature_alignment_report_master_iteration.csv')


### 1) Load Train/Test


In [2]:
train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)

print('Train shape:', train_df.shape)
print('Test shape :', test_df.shape)

missing_targets_train = [c for c in TARGET_COLS if c not in train_df.columns]
if missing_targets_train:
    raise RuntimeError(f'Missing target columns in train: {missing_targets_train}')

print('Train targets found:', [c for c in TARGET_COLS if c in train_df.columns])
print('Test targets found :', [c for c in TARGET_COLS if c in test_df.columns])


Train shape: (9319, 169)
Test shape : (200, 130)
Train targets found: ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
Test targets found : ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']


### 2) Load Or Build Feature Contract


In [3]:
if CONTRACT_TXT_PATH.exists():
    feature_contract_cols = [
        line.strip() for line in CONTRACT_TXT_PATH.read_text(encoding='utf-8').splitlines()
        if line.strip()
    ]
    print('Loaded feature contract from TXT:', CONTRACT_TXT_PATH)
else:
    # Fallback: derive contract directly from train and persist it.
    feature_contract_cols = [c for c in train_df.columns if c not in TARGET_COLS]
    CONTRACT_TXT_PATH.write_text('\n'.join(feature_contract_cols) + '\n', encoding='utf-8')
    print('Contract TXT not found; derived and wrote:', CONTRACT_TXT_PATH)

# De-duplicate while preserving order
feature_contract_cols = list(dict.fromkeys(feature_contract_cols))

# Hard guard: targets must not be in feature contract
leaky_targets = [c for c in feature_contract_cols if c in TARGET_COLS]
if leaky_targets:
    raise RuntimeError(f'Target leakage in contract: {leaky_targets}')

# Ensure train fully supports contract
missing_in_train = [c for c in feature_contract_cols if c not in train_df.columns]
if missing_in_train:
    raise RuntimeError(f'Contract columns missing in train: {missing_in_train[:20]}')

# Build/update meta snapshot to keep run reproducible
dtype_map = {c: str(train_df[c].dtype) for c in feature_contract_cols}
contract_meta = {
    'train_path': str(TRAIN_PATH),
    'test_path': str(TEST_PATH),
    'n_train_rows': int(len(train_df)),
    'n_test_rows': int(len(test_df)),
    'n_feature_columns': int(len(feature_contract_cols)),
    'target_columns': TARGET_COLS,
    'feature_columns': feature_contract_cols,
    'dtype_map': dtype_map,
    'sanlc_feature_count': int(sum(c.startswith(SANLC_PREFIXES) for c in feature_contract_cols)),
}
CONTRACT_META_PATH.write_text(json.dumps(contract_meta, ensure_ascii=False, indent=2), encoding='utf-8')

print('Feature contract size:', len(feature_contract_cols))
print('SANLC features in contract:', contract_meta['sanlc_feature_count'])
print('Contract meta saved:', CONTRACT_META_PATH)


Loaded feature contract from TXT: ..\data\interim\feature_contract_master_iteration.txt
Feature contract size: 166
SANLC features in contract: 130
Contract meta saved: ..\data\interim\feature_contract_master_iteration_meta.json


### 3) Align Test Schema To Train Contract


In [4]:
test_original_cols = test_df.columns.tolist()
test_original_set = set(test_original_cols)

missing_in_test = [c for c in feature_contract_cols if c not in test_df.columns]
extra_in_test = [c for c in test_df.columns if (c not in feature_contract_cols and c not in TARGET_COLS)]

print('Missing contract columns in test:', len(missing_in_test))
print('Extra non-target columns in test:', len(extra_in_test))

# Add missing columns to test with rule-based fill
fill_strategy = {}
for col in missing_in_test:
    if col.startswith(SANLC_PREFIXES):
        test_df[col] = 0.0
        fill_strategy[col] = 'zero_fill_sanlc_absent_class'
    else:
        test_df[col] = np.nan
        fill_strategy[col] = 'nan_fill_non_sanlc_missing'

# Drop extra non-target columns from test
test_df = test_df.drop(columns=extra_in_test, errors='ignore')

# Reorder feature block to match train contract exactly
train_feature_block = train_df[feature_contract_cols].copy()
test_feature_block = test_df.reindex(columns=feature_contract_cols).copy()

# Cast test dtypes to train dtypes where feasible
cast_failures = []
for col in feature_contract_cols:
    target_dtype = train_feature_block[col].dtype
    try:
        test_feature_block[col] = test_feature_block[col].astype(target_dtype)
    except Exception:
        cast_failures.append((col, str(target_dtype), str(test_feature_block[col].dtype)))

if cast_failures:
    print('dtype cast failures (kept original test dtype):', len(cast_failures))
    print('sample failures:', cast_failures[:10])

# Reattach targets (train always has, test may be NaN placeholders)
train_target_block = train_df[[c for c in TARGET_COLS if c in train_df.columns]].copy()
test_target_block = test_df[[c for c in TARGET_COLS if c in test_df.columns]].copy()

train_aligned = pd.concat([train_feature_block, train_target_block], axis=1)
test_aligned = pd.concat([test_feature_block, test_target_block], axis=1)

print('Aligned train shape:', train_aligned.shape)
print('Aligned test shape :', test_aligned.shape)


Missing contract columns in test: 42
Extra non-target columns in test: 3
Aligned train shape: (9319, 169)
Aligned test shape : (200, 169)


### 4) Validation Checks (Before Modeling)


In [5]:
# 4.1 Schema parity
assert list(train_aligned.columns[:len(feature_contract_cols)]) == feature_contract_cols
assert list(test_aligned.columns[:len(feature_contract_cols)]) == feature_contract_cols

# 4.2 Leakage guard
assert not any(c in TARGET_COLS for c in feature_contract_cols), 'Feature contract contains targets.'

# 4.3 Missingness + fill behavior summary
missing_after_alignment = test_aligned[feature_contract_cols].isna().sum()
nonzero_missing_after = missing_after_alignment[missing_after_alignment > 0]

# 4.4 SANLC sanity checks
sanlc_cols = [c for c in feature_contract_cols if c.startswith(SANLC_PREFIXES)]
train_sanlc_sum = train_aligned[sanlc_cols].sum(axis=1, skipna=True) if sanlc_cols else pd.Series(dtype=float)
test_sanlc_sum = test_aligned[sanlc_cols].sum(axis=1, skipna=True) if sanlc_cols else pd.Series(dtype=float)

# 4.5 Pseudo-spatial readiness check
geo_presence = {
    'train_has_geo_time': all(c in train_aligned.columns for c in GEO_TIME_COLS),
    'test_has_geo_time': all(c in test_aligned.columns for c in GEO_TIME_COLS),
}

report = {
    'train_rows': int(len(train_aligned)),
    'test_rows': int(len(test_aligned)),
    'contract_feature_count': int(len(feature_contract_cols)),
    'missing_in_test_before_alignment': int(len(missing_in_test)),
    'extra_in_test_before_drop': int(len(extra_in_test)),
    'sanlc_columns_in_contract': int(len(sanlc_cols)),
    'sanlc_missing_added_to_test': int(sum(c in fill_strategy for c in sanlc_cols)),
    'non_sanlc_missing_added_to_test': int(sum((c in fill_strategy) and (not c.startswith(SANLC_PREFIXES)) for c in feature_contract_cols)),
    'test_columns_still_with_nan_after_alignment': int((nonzero_missing_after > 0).sum()),
    'train_has_geo_time': geo_presence['train_has_geo_time'],
    'test_has_geo_time': geo_presence['test_has_geo_time'],
}

report_df = pd.DataFrame([report])
display(report_df)

if len(nonzero_missing_after) > 0:
    print('Columns still containing NaN in aligned test (top 30):')
    display(nonzero_missing_after.sort_values(ascending=False).head(30).to_frame('nan_count'))

if len(sanlc_cols) > 0:
    sanlc_sum_summary = pd.DataFrame({
        'dataset': ['train', 'test_aligned'],
        'mean_row_sum': [float(train_sanlc_sum.mean()), float(test_sanlc_sum.mean())],
        'p05_row_sum': [float(train_sanlc_sum.quantile(0.05)), float(test_sanlc_sum.quantile(0.05))],
        'p50_row_sum': [float(train_sanlc_sum.quantile(0.50)), float(test_sanlc_sum.quantile(0.50))],
        'p95_row_sum': [float(train_sanlc_sum.quantile(0.95)), float(test_sanlc_sum.quantile(0.95))],
    })
    print('SANLC row-sum sanity summary:')
    display(sanlc_sum_summary)

if not geo_presence['train_has_geo_time'] or not geo_presence['test_has_geo_time']:
    print('WARNING: geo/time keys missing. Pseudo-spatial split cannot be trusted until these keys exist in both datasets.')
else:
    print('Geo/time keys present in both datasets. Pseudo-spatial split is ready for model selection workflow.')


,train_rows,test_rows,contract_feature_count,missing_in_test_before_alignment,extra_in_test_before_drop,sanlc_columns_in_contract,sanlc_missing_added_to_test,non_sanlc_missing_added_to_test,test_columns_still_with_nan_after_alignment,train_has_geo_time,test_has_geo_time
0,9319,200,166,42,3,130,37,5,11,True,True


Columns still containing NaN in aligned test (top 30):


,nan_count
weather_precip_7d_sum,200
weather_temp_7d_max,200
dem_elev_mean_1km,200
weather_wind_7d_mean,200
dem_slope_1km,200
nir,19
green,19
swir16,19
MNDWI,19
swir22,19


SANLC row-sum sanity summary:


,dataset,mean_row_sum,p05_row_sum,p50_row_sum,p95_row_sum
0,train,2.0,2.0,2.0,2.0
1,test_aligned,2.0,2.0,2.0,2.0


Geo/time keys present in both datasets. Pseudo-spatial split is ready for model selection workflow.


### 5) Save Aligned Artifacts


In [6]:
OUT_TRAIN_PATH.parent.mkdir(parents=True, exist_ok=True)

train_aligned.to_parquet(OUT_TRAIN_PATH, index=False)
test_aligned.to_parquet(OUT_TEST_PATH, index=False)

# Column-level alignment report
rows = []
for col in feature_contract_cols:
    rows.append({
        'feature': col,
        'in_train': col in train_df.columns,
        'in_test_original': col in test_original_set,
        'added_to_test': col in fill_strategy,
        'fill_strategy': fill_strategy.get(col, 'none'),
        'train_dtype': str(train_df[col].dtype) if col in train_df.columns else None,
        'test_dtype_aligned': str(test_aligned[col].dtype) if col in test_aligned.columns else None,
        'test_nan_count_aligned': int(test_aligned[col].isna().sum()) if col in test_aligned.columns else None,
    })

alignment_report_df = pd.DataFrame(rows)
alignment_report_df.to_csv(OUT_REPORT_PATH, index=False)

print('Saved:')
print('-', OUT_TRAIN_PATH)
print('-', OUT_TEST_PATH)
print('-', OUT_REPORT_PATH)
print('Done.')


Saved:
- ..\data\interim\master_train_iteration_aligned.parquet
- ..\data\interim\master_test_iteration_aligned.parquet
- ..\data\interim\feature_alignment_report_master_iteration.csv
Done.
